In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage.measure import regionprops, regionprops_table
from keras.utils import load_img
from keras.saving import load_model
from importlib import reload
import segmenteverygrain as seg
import sez
from segment_anything import sam_model_registry, SamPredictor
from tqdm import trange, tqdm
import os
import geopandas as gpd
import rasterio
%matplotlib qt
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import rasterio
from shapely.geometry import Polygon
# create geopandas dataframe
import geopandas
import tifffile
from shapely import wkt
from rasterio.features import rasterize
# Assume you have your original image loaded to get shape:
import cv2

# Extracting morphometric information from segmented grains
### 
- this notebook is intended to be run after all grains have been segmented, using either the create_polygons_and_masks.ipynb file or otherwise

### Step 1: Read in the original image

In [ ]:
original_image = cv2.imread('/Users/omw339/Desktop/OWBP25012/OWBP25012/OWBP25012.tif')

### Step 2: Read in the Csv file with the coordinates of the segmented polygon to re-initalize the geodataframe

In [ ]:
# Replace with path to your csv file of the segmented polygon coordinates
csv_path = '/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25001/OWBP25001_draft_coordinates_9_16_25_final.csv'

In [ ]:
height, width = original_image.shape[:2]

# Create shapes generator: (geometry, label) tuples
gdf = sez.load_polygons("path/to/your/file.csv", crs="EPSG:4326")
shapes = ((geom, idx + 1) for idx, geom in enumerate(gdf.geometry))

# Rasterize polygons:
label_image = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    fill=0,
    dtype=np.uint16
)

print(f"Rasterized label image shape: {label_image.shape}")
print(f"Max label value (should equal number of polygons): {label_image.max()}")

Ensure that the max label value is equivalent to the number of grains you segmented, otherwise double-check your csv file

In [ ]:
tifffile.imwrite('/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25012/OWBP25012_draft_10_04_rasterized_labels.tif', label_image)

### Step 3: Creating the morphometrics dataframe.
- The following cell creates the dataframe for the morphometric information. It is important to note that the original columns are in **pixels**

In [ ]:
from skimage.measure import regionprops_table

# Define the properties you want
properties = [
    'label',
    'area',
    'centroid',
    'bbox',
    'major_axis_length',
    'minor_axis_length',
    'eccentricity',
    'solidity',
    'orientation',
    'perimeter'
]

# Extract properties for each labeled region (grain)
props = regionprops_table(label_image, properties=properties)

# Convert to DataFrame
grain_data = pd.DataFrame(props)

print(grain_data.head())


### Step 4: The following cells are used to get morphometric information in micrometers (or any other unit)

**Option 1**: Use the length of the scale bar in pixels to get the scale of the image (in units / pixel) within the notebook. Run this cell and then click (left mouse button) on one end of the scale bar in the image and click (right mouse button) on the other end of the scale bar:

In [ ]:
cid5 = fig.canvas.mpl_connect('button_press_event', lambda event: seg.click_for_scale(event, ax))

**Option 2 (Recommended)**: Open up the original image in any image processing software such as FIJI or ImageJ to measure the scalebar

- n_of_units: represents the number on the scale bar in the image
- scale_bar_length: the length of the scale bar in pixels that you measured


In [ ]:
n_of_units = 1000 # micrometers usually'
scale_bar_length = 1802.167 #length of scale bar in pixels
units_per_pixel = n_of_units/scale_bar_length

Adding columns to the grain_data dataframe for the morphometric measurements in the scaled measurement of your choice 

In [ ]:

grain_data['area_micron2'] = grain_data['area'] * (units_per_pixel ** 2)
grain_data['perimeter_micron'] = grain_data['perimeter'] * units_per_pixel
grain_data['major_axis_length_micron'] = grain_data['major_axis_length'] * units_per_pixel
grain_data['minor_axis_length_micron'] = grain_data['minor_axis_length'] * units_per_pixel

# For centroid and bbox, convert all coordinate values:
grain_data['centroid_row_micron'] = grain_data['centroid-0'] * units_per_pixel
grain_data['centroid_col_micron'] = grain_data['centroid-1'] * units_per_pixel

grain_data['bbox_min_row_micron'] = grain_data['bbox-0'] * units_per_pixel
grain_data['bbox_min_col_micron'] = grain_data['bbox-1'] * units_per_pixel
grain_data['bbox_max_row_micron'] = grain_data['bbox-2'] * units_per_pixel
grain_data['bbox_max_col_micron'] = grain_data['bbox-3'] * units_per_pixel

In [ ]:
print(grain_data)

saving morphometric dataframe to csv

In [ ]:
grain_data.to_csv('/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25001/OWBP25001_draft_9_16_morphometrics.csv')